Install libraries and upload Excel

In [5]:
!pip -q install openpyxl beautifulsoup4

import pandas as pd
import re
import os
import shutil
import unicodedata
from bs4 import BeautifulSoup
from datetime import datetime
from google.colab import files

uploaded = files.upload()

xlsx_files = [
    name for name in uploaded
    if name.lower().endswith(".xlsx")
]

print("Uploaded Excel files:")
for f in xlsx_files:
    print(" -", f)

# Automatically identify sender_place_new.xlsx
SENDER_PLACE_FILE = next(
    f for f in xlsx_files
    if "sender_place_new" in f.lower()
)

# Everything else is treated as the main metadata Excel
INPUT_FILE = next(
    f for f in xlsx_files
    if f != SENDER_PLACE_FILE
)

df = pd.read_excel(
    INPUT_FILE,
    dtype=object
)

sender_place_df = pd.read_excel(
    SENDER_PLACE_FILE,
    dtype=object
)

print("\nMAIN EXCEL")
print("File:", INPUT_FILE)
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nSENDER PLACE MAPPING")
print("File:", SENDER_PLACE_FILE)
print("Rows:", len(sender_place_df))
print("Columns:", list(sender_place_df.columns))

display(
    sender_place_df[
        ["sender_place", "sender_place_new"]
    ].head(10)
)

Saving franckenachlass_8.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx to franckenachlass_8.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx
Saving sender_place_new.xlsx to sender_place_new.xlsx
Uploaded Excel files:
 - franckenachlass_8.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx
 - sender_place_new.xlsx

MAIN EXCEL
File: franckenachlass_8.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx
Rows: 324
Columns: 89

SENDER PLACE MAPPING
File: sender_place_new.xlsx
Rows: 124
Columns: ['no', 'sender_place', 'sender_place_new', 'sender_place_new_version1', 'no.1', 'Unnamed: 5', 'stripped ', 'church/ congregation', 'street ', 'Unnamed: 9']


,sender_place,sender_place_new
0,"[Amsterdam, North Holland, Netherlands]","[Amsterdam, Netherlands]"
1,"[Berlin, Germany]","[Berlin, Germany]"
2,"[Frankfurt, Hesse, Germany]","[Frankfurt, Germany]"
3,"[Frutigen, Bern, Switzerland]","[Frutigen, Switzerland]"
4,"[Offstein, Germany]","[Offstein, Germany]"
5,"[Palatinate, Germany]","[Palatinate, Germany]"
6,"[Sicily, Italy]","[Sicily, Italy]"
7,"Aachen, North Rhine-Westphalia, Germany","Aachen, Germany"
8,"Alkmaar, North Holland, Netherlands","Alkmaar, Netherlands"
9,"Alsace, France","Alsace, France"


Convert Excel → MarcEdit .mrk

In [7]:
import pandas as pd
import re
import os
import shutil
import unicodedata
from bs4 import BeautifulSoup
from datetime import datetime
from google.colab import files


# ============================================================
# OUTPUT
# ============================================================

OUTPUT_MRK = "converted_records_updated.mrk"

LOG_DIR = "marc_field_logs"
MASTER_LOG = "MARC_MASTER_LOG.csv"
SUMMARY_LOG = "MARC_LOG_SUMMARY.csv"

os.makedirs(LOG_DIR, exist_ok=True)


# ============================================================
# RUN VALUES
# ============================================================

RUN_TIME = datetime.now()

RUN_005 = RUN_TIME.strftime("%Y%m%d%H%M%S")
RUN_008_DATE = RUN_TIME.strftime("%y%m%d")


# ============================================================
# CONSTANTS
# ============================================================

DEFAULT_LEADER = "00000ckm a2200000 io4500"

PLACEHOLDER_001 = " "
PLACEHOLDER_024 = "PLACEHOLDER"
#default value if 035 doesn't exist in which 024 takes value from

FIXED_040 = "$aFTU$beng$erda$cFTU$dFTU"
FIXED_043 = "$ae-uk-en"
FIXED_049 = "$a[elec]FTUU"

FIXED_336 = "$astill image$bsti$2rdacontent"
FIXED_337 = "$aunmediated$bn$2rdamedia"
FIXED_338 = "$asheet$bnb$2rdacarrier"

DEFAULT_CALC_URL = "https://stars.library.ucf.edu"

CALC_LINK_TEXT = (
    "Click here for digital images of the original letter"
)


# ============================================================
# BASIC HELPERS
# ============================================================

def clean(value):

    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    value = str(value).strip()

    if value.lower() in {
        "nan",
        "none",
        "nat"
    }:
        return ""

    return value


def normalize_space(value):

    return re.sub(
        r"\s+",
        " ",
        clean(value)
    ).strip()


def strip_html(value):



    value = clean(value)



    if not value:

        return ""



    soup = BeautifulSoup(

        value,

        "html.parser"

    )

    return normalize_space(
        soup.get_text(
            " ",
            strip=True
        )
    )


def mf(
    tag,
    ind1="\\",
    ind2="\\",
    body=""
):

    return f"={tag}  {ind1}{ind2}{body}"


# ============================================================
# LOGGING
# ============================================================

LOGS = {}


def log_event(
    tag,
    record_number,
    excel_row,
    status,
    source_column="",
    source_value="",
    output_value="",
    reason="",
    instance=1
):

    tag = str(tag)

    if tag not in LOGS:
        LOGS[tag] = []

    LOGS[tag].append({
        "record_number": record_number,
        "excel_row": excel_row,
        "marc_field": tag,
        "instance": instance,
        "status": status,
        "source_column": source_column,
        "source_value": clean(source_value),
        "output_value": output_value,
        "reason": reason
    })


def log_skip(
    tag,
    record_number,
    excel_row,
    source_column="",
    source_value="",
    reason=""
):

    log_event(
        tag,
        record_number,
        excel_row,
        "SKIPPED",
        source_column,
        source_value,
        "",
        reason
    )


def add_output(
    lines,
    tag,
    record_number,
    excel_row,
    value,
    status,
    source_column,
    source_value,
    reason,
    instance=1
):

    lines.append(value)

    log_event(
        tag,
        record_number,
        excel_row,
        status,
        source_column,
        source_value,
        value,
        reason,
        instance
    )


# ============================================================
# NORMALIZED MATCHING
# ============================================================

def norm_match(value):

    value = clean(value)

    value = unicodedata.normalize(
        "NFKD",
        value
    )

    value = "".join(
        x for x in value
        if not unicodedata.combining(x)
    )

    value = value.lower()

    value = re.sub(
        r"\(q\d+\)",
        "",
        value
    )

    value = re.sub(
        r"[^a-z0-9]+",
        " ",
        value
    )

    return " ".join(
        value.split()
    )


def remove_qid(label):

    return re.sub(
        r"\s*\(Q\d+\)\s*$",
        "",
        normalize_space(label),
        flags=re.I
    ).strip()


# ============================================================
# SENDER PLACE MAPPING
# ============================================================

def normalize_sender_place_key(value):

    value = clean(value)

    if not value:
        return ""

    if (
        value.startswith("[")
        and value.endswith("]")
    ):
        value = value[1:-1]

    value = normalize_space(value)

    value = re.sub(
        r"\s*,\s*",
        ", ",
        value
    )

    value = re.sub(
        r"\s*;\s*",
        "; ",
        value
    )

    return value.casefold()


# sender_place_df comes from Cell 1

SENDER_PLACE_LOOKUP = {}

for _, map_row in sender_place_df.iterrows():

    original = clean(
        map_row.get(
            "sender_place",
            ""
        )
    )

    updated = clean(
        map_row.get(
            "sender_place_new",
            ""
        )
    )

    if not original:
        continue

    key = normalize_sender_place_key(
        original
    )

    if key:
        SENDER_PLACE_LOOKUP[key] = updated


def get_updated_sender_place(original_value):

    original_value = clean(
        original_value
    )

    if not original_value:

        return (
            "",
            "SKIPPED",
            "sender_place empty"
        )

    key = normalize_sender_place_key(
        original_value
    )

    if key in SENDER_PLACE_LOOKUP:

        updated = clean(
            SENDER_PLACE_LOOKUP[key]
        )

        if updated:

            return (
                updated,
                "REPLACED",
                (
                    "Matched sender_place against "
                    "sender_place_new.xlsx and used "
                    "sender_place_new"
                )
            )

    return (
        original_value,
        "FALLBACK_ORIGINAL",
        (
            "No replacement found in "
            "sender_place_new.xlsx; original "
            "sender_place retained"
        )
    )


# ============================================================
# LINKED_NEW PRIORITY
# ============================================================

def get_linked(
    row,
    new_column,
    old_column
):

    new_value = clean(
        row.get(
            new_column,
            ""
        )
    )

    old_value = clean(
        row.get(
            old_column,
            ""
        )
    )

    if new_value:

        if (
            old_value
            and new_value != old_value
        ):

            return (
                new_value,
                new_column,
                "REPLACED",
                (
                    f"{new_column} replaced "
                    f"{old_column}"
                )
            )

        return (
            new_value,
            new_column,
            "USED_NEW",
            f"Preferred {new_column}"
        )

    if old_value:

        return (
            old_value,
            old_column,
            "FALLBACK",
            (
                f"{new_column} empty; "
                f"used {old_column}"
            )
        )

    return (
        "",
        "",
        "SKIPPED",
        (
            f"Both {new_column} and "
            f"{old_column} empty"
        )
    )


# ============================================================
# DATES
# ============================================================

def year_from_value(value):

    match = re.search(
        r"\b(1[0-9]{3}|20[0-9]{2})\b",
        clean(value)
    )

    return (
        match.group(1)
        if match
        else ""
    )


def normalize_yyyymmdd(value):

    value = clean(value)

    if not value:
        return ""

    match = re.search(
        r"(1[0-9]{3}|20[0-9]{2})"
        r"[-/.](\d{1,2})"
        r"[-/.](\d{1,2})",
        value
    )

    if match:

        return (
            match.group(1)
            + match.group(2).zfill(2)
            + match.group(3).zfill(2)
        )

    match = re.search(
        r"(\d{1,2})"
        r"[-/.](\d{1,2})"
        r"[-/.](1[0-9]{3}|20[0-9]{2})",
        value
    )

    if match:

        return (
            match.group(3)
            + match.group(1).zfill(2)
            + match.group(2).zfill(2)
        )

    digits = re.sub(
        r"\D",
        "",
        value
    )

    if len(digits) == 8:
        return digits

    return ""


# ============================================================
# LANGUAGE
# ============================================================

LANGUAGE_MAP = {

    "english": "eng",
    "eng": "eng",

    "dutch": "dut",
    "dut": "dut",
    "nld": "dut",

    "german": "ger",
    "ger": "ger",
    "deu": "ger",

    "french": "fre",
    "fre": "fre",
    "fra": "fre",

    "spanish": "spa",
    "spa": "spa",

    "latin": "lat",
    "lat": "lat"
}


def get_language_code(value):

    value = clean(value).lower()

    if not value:
        return "eng"

    return LANGUAGE_MAP.get(
        value,
        value[:3]
    )


def build_008(
    year,
    language
):

    if not re.fullmatch(
        r"\d{4}",
        year or ""
    ):
        year = "||||"

    language = get_language_code(
        language
    )

    return (
        RUN_008_DATE
        + "s"
        + year
        + ("\\" * 4)
        + "flunnn"
        + ("\\" * 12)
        + "kn"
        + language
        + "\\d"
    )


# ============================================================
# 024
# ============================================================

def extract_024(custom_citation):

    citation = clean(
        custom_citation
    )

    if not citation:
        return ""

    match = re.search(
        r"\bPRINT\s*,\s*(.+?)\s*$",
        citation,
        flags=re.I
    )

    if not match:
        return ""

    return match.group(1).strip()


# ============================================================
# SIMPLE MULTI VALUES
# ============================================================

def split_values(value):

    value = clean(value)

    if not value:
        return []

    return [
        normalize_space(x)
        for x in re.split(
            r"\s*;\s*|\r?\n+",
            value
        )
        if normalize_space(x)
    ]


# ============================================================
# AUTHORITY PARSING
# ============================================================

def authority_source(
    url,
    surrounding_text=""
):

    url_lower = clean(
        url
    ).lower()

    text_lower = clean(
        surrounding_text
    ).lower()

    if "id.loc.gov" in url_lower:
        return "LC"

    if "viaf.org" in url_lower:
        return "VIAF"

    if "wikidata.org" in url_lower:
        return "Wikidata"

    if "geonames.org" in url_lower:
        return "GeoNames"

    if "[lc]" in text_lower:
        return "LC"

    if "[viaf]" in text_lower:
        return "VIAF"

    if "[wikidata]" in text_lower:
        return "Wikidata"

    if "[geonames]" in text_lower:
        return "GeoNames"

    return "Unknown"


def parse_authority_links(value):

    value = clean(value)

    if not value:
        return []

    soup = BeautifulSoup(
        value,
        "html.parser"
    )

    results = []

    for anchor in soup.find_all("a"):

        label = normalize_space(
            anchor.get_text(
                " ",
                strip=True
            )
        )

        url = clean(
            anchor.get(
                "href",
                ""
            )
        ).rstrip("/")

        parent_text = normalize_space(
            anchor.parent.get_text(
                " ",
                strip=True
            )
        )

        results.append({
            "label": remove_qid(label),
            "url": url,
            "source": authority_source(
                url,
                parent_text
            )
        })

    if not results:

        text = strip_html(
            value
        )

        if text:

            results.append({
                "label": text,
                "url": "",
                "source": "Unknown"
            })

    return results


# ============================================================
# STRICT ENTITY GROUPING
#
# IMPORTANT NEWEST UPDATE
#
# Authority records for the SAME creator are grouped.
#
# Example:
#
# LC Person A
# VIAF Person A
# Wikidata Person A
#
# -> ONE creator entity.
#
# LC Person B
# VIAF Person B
#
# -> SECOND creator entity -> 700.
#
# ============================================================

def normalized_entity_name(label):

    label = remove_qid(
        label
    )

    # Remove dates
    label = re.sub(
        r",?\s*(?:ca\.\s*)?"
        r"-?\d{4}(?:-\d{0,4})?\s*$",
        "",
        label,
        flags=re.I
    )

    label = label.replace(
        ",",
        " "
    )

    return norm_match(
        label
    )


def names_probably_same(
    label1,
    label2
):

    n1 = normalized_entity_name(
        label1
    )

    n2 = normalized_entity_name(
        label2
    )

    if not n1 or not n2:
        return False

    if n1 == n2:
        return True

    tokens1 = n1.split()
    tokens2 = n2.split()

    if sorted(tokens1) == sorted(tokens2):
        return True

    # Handle extra middle names / particles cautiously
    set1 = set(tokens1)
    set2 = set(tokens2)

    if (
        len(set1) >= 2
        and len(set2) >= 2
    ):

        overlap = (
            len(set1 & set2)
            / max(
                len(set1),
                len(set2)
            )
        )

        if overlap >= 0.75:
            return True

    return False


def group_authorities(value):

    links = parse_authority_links(
        value
    )

    if not links:
        return []

    entities = []

    for link in links:

        matched_entity = None

        for entity in entities:

            # Do not attach two LC records to one person.
            # Repeated LC normally means another creator.
            if (
                link["source"] == "LC"
                and any(
                    x["source"] == "LC"
                    for x in entity["links"]
                )
            ):
                continue

            if names_probably_same(
                link["label"],
                entity["label"]
            ):

                matched_entity = entity
                break

        if matched_entity is not None:

            matched_entity[
                "links"
            ].append(
                link
            )

            # Upgrade canonical label based on authority priority
            existing_priority = {
                "LC": 1,
                "VIAF": 2,
                "Wikidata": 3,
                "GeoNames": 4,
                "Unknown": 5
            }

            current_source = (
                matched_entity[
                    "canonical_source"
                ]
            )

            if (
                existing_priority.get(
                    link["source"],
                    99
                )
                <
                existing_priority.get(
                    current_source,
                    99
                )
            ):

                matched_entity[
                    "label"
                ] = link[
                    "label"
                ]

                matched_entity[
                    "canonical_source"
                ] = link[
                    "source"
                ]

        else:

            entities.append({
                "label": link["label"],
                "canonical_source": link["source"],
                "links": [link]
            })

    return entities


def get_link(
    entity,
    source
):

    return next(
        (
            x
            for x in entity[
                "links"
            ]
            if x[
                "source"
            ] == source
        ),
        None
    )


# ============================================================
# STRUCTURED AUTHORS
# ============================================================

def get_structured_people(row):

    people = []

    for i in range(
        1,
        6
    ):

        fname = clean(
            row.get(
                f"author{i}_fname",
                ""
            )
        )

        mname = clean(
            row.get(
                f"author{i}_mname",
                ""
            )
        )

        lname = clean(
            row.get(
                f"author{i}_lname",
                ""
            )
        )

        suffix = clean(
            row.get(
                f"author{i}_suffix",
                ""
            )
        )

        corporate = clean(
            row.get(
                f"author{i}_is_corporate",
                ""
            )
        ).lower()

        if not any([
            fname,
            mname,
            lname
        ]):
            continue

        direct = " ".join(
            x for x in [
                fname,
                mname,
                lname
            ]
            if x
        )

        given = " ".join(
            x for x in [
                fname,
                mname
            ]
            if x
        )

        if lname and given:

            inverted = (
                lname
                + ", "
                + given
            )

        elif lname:

            inverted = lname

        else:

            inverted = given

        people.append({
            "index": i,
            "fname": fname,
            "mname": mname,
            "lname": lname,
            "suffix": suffix,
            "direct": direct,
            "inverted": inverted,
            "is_corporate":
                corporate in {
                    "1",
                    "true",
                    "yes",
                    "y"
                }
        })

    return people


def find_structured_person(
    row,
    label
):

    label_normalized = norm_match(
        label
    )

    for person in get_structured_people(
        row
    ):

        if person[
            "is_corporate"
        ]:
            continue

        if label_normalized in {
            norm_match(
                person["direct"]
            ),
            norm_match(
                person["inverted"]
            )
        }:

            return person

    return None


# ============================================================
# NAME / DATE
# ============================================================

DATE_PATTERN = (
    r"(?:ca\.\s*)?"
    r"-?\d{4}"
    r"(?:-\d{0,4})?"
)


def extract_trailing_date(label):

    label = normalize_space(
        label
    )

    match = re.search(
        rf",?\s*({DATE_PATTERN})\s*$",
        label,
        flags=re.I
    )

    if not match:

        return (
            label.rstrip(" ,"),
            ""
        )

    date = match.group(1).strip()

    name = label[
        :match.start()
    ].rstrip(" ,")

    return (
        name,
        date
    )


def direct_to_inverted(
    name,
    row=None
):

    name = normalize_space(
        remove_qid(
            name
        )
    )

    if not name:
        return "", "FAILED"

    if "," in name:

        return (
            name,
            "ALREADY_INVERTED"
        )

    if row is not None:

        person = find_structured_person(
            row,
            name
        )

        if (
            person
            and person[
                "lname"
            ]
        ):

            result = person[
                "inverted"
            ]

            if person[
                "suffix"
            ]:

                result += (
                    ", "
                    + person[
                        "suffix"
                    ]
                )

            return (
                result,
                "STRUCTURED_AUTHOR"
            )

    tokens = name.split()

    if len(tokens) < 2:

        return (
            name,
            "HEURISTIC"
        )

    return (
        tokens[-1]
        + ", "
        + " ".join(
            tokens[:-1]
        ),
        "HEURISTIC"
    )


# ============================================================
# BUILD PERSONAL NAME FIELD
# ============================================================

def build_person_body(
    entity,
    row,
    role
):

    lc = get_link(
        entity,
        "LC"
    )

    viaf = get_link(
        entity,
        "VIAF"
    )

    wiki = get_link(
        entity,
        "Wikidata"
    )

    # --------------------------------------------------------
    # LC PREFERRED
    # --------------------------------------------------------

    if lc:

        name, date = extract_trailing_date(
            lc["label"]
        )

        ind1 = (
            "1"
            if "," in name
            else "0"
        )

        body = (
            "$a"
            + name
        )

        if date:

            body += (
                ",$d"
                + date
            )

        if role:

            body += (
                ",$e"
                + role
            )

        if lc["url"]:

            body += (
                "$0"
                + lc["url"]
            )

        if (
            viaf
            and viaf["url"]
        ):

            body += (
                "$1"
                + viaf["url"]
            )

        if (
            wiki
            and wiki["url"]
        ):

            body += (
                "$1"
                + wiki["url"]
            )

        return (
            body,
            ind1,
            "LC"
        )


    # --------------------------------------------------------
    # VIAF SECOND
    # --------------------------------------------------------

    if viaf:

        name, date = extract_trailing_date(
            viaf["label"]
        )

        method = "VIAF"

        if "," not in name:

            name, method = direct_to_inverted(
                name,
                row
            )

        body = (
            "$a"
            + name
        )

        if date:

            body += (
                ",$d"
                + date
            )

        if role:

            body += (
                ",$e"
                + role
            )

        if viaf["url"]:

            body += (
                "$1"
                + viaf["url"]
            )

        if (
            wiki
            and wiki["url"]
        ):

            body += (
                "$1"
                + wiki["url"]
            )

        return (
            body,
            "1",
            (
                "HEURISTIC"
                if method
                == "HEURISTIC"
                else "VIAF"
            )
        )


    # --------------------------------------------------------
    # WIKIDATA THIRD
    # --------------------------------------------------------

    if wiki:

        name, date = extract_trailing_date(
            wiki["label"]
        )

        name, method = direct_to_inverted(
            name,
            row
        )

        body = (
            "$a"
            + name
        )

        if date:

            body += (
                ",$d"
                + date
            )

        if role:

            body += (
                ",$e"
                + role
            )

        if wiki["url"]:

            body += (
                "$1"
                + wiki["url"]
            )

        return (
            body,
            "1",
            (
                "HEURISTIC"
                if method
                == "HEURISTIC"
                else "WIKIDATA"
            )
        )


    # --------------------------------------------------------
    # PLAIN
    # --------------------------------------------------------

    name, date = extract_trailing_date(
        entity["label"]
    )

    body = (
        "$a"
        + name
    )

    if date:

        body += (
            ",$d"
            + date
        )

    if role:

        body += (
            ",$e"
            + role
        )

    return (
        body,
        (
            "1"
            if "," in name
            else "0"
        ),
        "PLAIN"
    )


# ============================================================
# DISPLAY NAME FOR 245 / 246
# ============================================================

def direct_display_name(name):

    name, _ = extract_trailing_date(
        name
    )

    if "," not in name:
        return name.strip()

    parts = [
        x.strip()
        for x in name.split(
            ",",
            1
        )
    ]

    return (
        parts[1]
        + " "
        + parts[0]
    ).strip()


def get_creator_display_name(
    row,
    creator_entities
):

    people = [
        x
        for x in get_structured_people(
            row
        )
        if not x[
            "is_corporate"
        ]
    ]

    if people:

        return people[0][
            "direct"
        ]

    if creator_entities:

        return direct_display_name(
            creator_entities[
                0
            ][
                "label"
            ]
        )

    return ""


# ============================================================
# 600 PERSONAL SUBJECT
# ============================================================

def build_subject_person(
    entity
):

    lc = get_link(
        entity,
        "LC"
    )

    viaf = get_link(
        entity,
        "VIAF"
    )

    wiki = get_link(
        entity,
        "Wikidata"
    )

    if lc:

        name, date = extract_trailing_date(
            lc["label"]
        )

        body = (
            "$a"
            + name
        )

        if date:

            body += (
                ",$d"
                + date
            )

        if lc["url"]:

            body += (
                "$0"
                + lc["url"]
            )

        if (
            viaf
            and viaf["url"]
        ):

            body += (
                "$1"
                + viaf["url"]
            )

        if (
            wiki
            and wiki["url"]
        ):

            body += (
                "$1"
                + wiki["url"]
            )

        return (
            "1",
            "0",
            body,
            "LC"
        )

    chosen = (
        viaf
        or wiki
    )

    if chosen:

        name, date = extract_trailing_date(
            chosen["label"]
        )

        body = (
            "$a"
            + name
        )

        if date:

            body += (
                ",$d"
                + date
            )

        if (
            viaf
            and viaf["url"]
        ):

            body += (
                "$1"
                + viaf["url"]
            )

        if (
            wiki
            and wiki["url"]
        ):

            body += (
                "$1"
                + wiki["url"]
            )

        return (
            "0",
            "4",
            body,
            (
                "VIAF"
                if viaf
                else "Wikidata"
            )
        )

    return (
        "0",
        "4",
        (
            "$a"
            + entity[
                "label"
            ]
        ),
        "PLAIN"
    )


# ============================================================
# 610 CORPORATE
# ============================================================

def build_corporate_subject(
    entity
):

    lc = get_link(
        entity,
        "LC"
    )

    viaf = get_link(
        entity,
        "VIAF"
    )

    wiki = get_link(
        entity,
        "Wikidata"
    )

    if lc:

        body = (
            "$a"
            + lc[
                "label"
            ]
        )

        if lc["url"]:

            body += (
                "$0"
                + lc[
                    "url"
                ]
            )

        if (
            viaf
            and viaf["url"]
        ):

            body += (
                "$1"
                + viaf[
                    "url"
                ]
            )

        if (
            wiki
            and wiki["url"]
        ):

            body += (
                "$1"
                + wiki[
                    "url"
                ]
            )

        return (
            "2",
            "0",
            body,
            "LC"
        )

    chosen = (
        viaf
        or wiki
    )

    if chosen:

        body = (
            "$a"
            + chosen[
                "label"
            ]
        )

        if (
            viaf
            and viaf["url"]
        ):

            body += (
                "$1"
                + viaf[
                    "url"
                ]
            )

        if (
            wiki
            and wiki["url"]
        ):

            body += (
                "$1"
                + wiki[
                    "url"
                ]
            )

        if wiki:

            body += "$2wikidata"

        elif viaf:

            body += "$2viaf"

        return (
            "2",
            "4",
            body,
            (
                "Wikidata"
                if wiki
                else "VIAF"
            )
        )

    return (
        "2",
        "4",
        (
            "$a"
            + entity[
                "label"
            ]
        ),
        "PLAIN"
    )


# ============================================================
# 600 FAMILY
# ============================================================

def build_family_subject(
    entity
):

    lc = get_link(
        entity,
        "LC"
    )

    viaf = get_link(
        entity,
        "VIAF"
    )

    wiki = get_link(
        entity,
        "Wikidata"
    )

    if lc:

        body = (
            "$a"
            + lc[
                "label"
            ]
        )

        if lc["url"]:

            body += (
                "$0"
                + lc[
                    "url"
                ]
            )

        if (
            viaf
            and viaf["url"]
        ):

            body += (
                "$1"
                + viaf[
                    "url"
                ]
            )

        if (
            wiki
            and wiki["url"]
        ):

            body += (
                "$1"
                + wiki[
                    "url"
                ]
            )

        return (
            "3",
            "0",
            body,
            "LC"
        )

    chosen = (
        viaf
        or wiki
    )

    if chosen:

        body = (
            "$a"
            + chosen[
                "label"
            ]
        )

        if (
            viaf
            and viaf["url"]
        ):

            body += (
                "$1"
                + viaf[
                    "url"
                ]
            )

        if (
            wiki
            and wiki["url"]
        ):

            body += (
                "$1"
                + wiki[
                    "url"
                ]
            )

        return (
            "3",
            "4",
            body,
            (
                "VIAF"
                if viaf
                else "Wikidata"
            )
        )

    return (
        "3",
        "4",
        (
            "$a"
            + entity[
                "label"
            ]
        ),
        "PLAIN"
    )


# ============================================================
# ORGANIZATION RECEIVER MATCH
# ============================================================

def organization_matches_receiver(
    receiver,
    entity
):

    receiver_normalized = norm_match(
        receiver
    )

    organization_normalized = norm_match(
        entity[
            "label"
        ]
    )

    if not (
        receiver_normalized
        and organization_normalized
    ):
        return False

    return (
        receiver_normalized
        == organization_normalized
        or receiver_normalized
        in organization_normalized
        or organization_normalized
        in receiver_normalized
    )


# ============================================================
# MAIN RECORD CONVERTER
# ============================================================

def convert_row(
    row,
    record_number
):

    excel_row = (
        record_number
        + 1
    )

    lines = []


    # ========================================================
    # LDR
    # ========================================================

    value = (
        "=LDR  "
        + DEFAULT_LEADER
    )

    add_output(
        lines,
        "LDR",
        record_number,
        excel_row,
        value,
        "GENERATED",
        "updated documentation",
        DEFAULT_LEADER,
        "Updated leader"
    )


    # ========================================================
    # 001
    # ========================================================

    value = (
        "=001  "
        + PLACEHOLDER_001
    )

    add_output(
        lines,
        "001",
        record_number,
        excel_row,
        value,
        "PLACEHOLDER",
        "system",
        "",
        "Placeholder space"
    )


    # ========================================================
    # 005
    # ========================================================

    value = (
        "=005  "
        + RUN_005
    )

    add_output(
        lines,
        "005",
        record_number,
        excel_row,
        value,
        "GENERATED",
        "system datetime",
        RUN_005,
        "Same datetime for all records"
    )


    # ========================================================
    # YEAR
    # ========================================================

    year = clean(
        row.get(
            "date_year",
            ""
        )
    )

    if not re.fullmatch(
        r"\d{4}",
        year
    ):

        year = year_from_value(
            row.get(
                "publication_date",
                ""
            )
        )


    # ========================================================
    # LANGUAGE
    # ========================================================

    language_raw = clean(
        row.get(
            "language",
            ""
        )
    )

    language = get_language_code(
        language_raw
    )


    # ========================================================
    # 008
    # ========================================================

    value = (
        "=008  "
        + build_008(
            year,
            language_raw
        )
    )

    add_output(
        lines,
        "008",
        record_number,
        excel_row,
        value,
        "GENERATED",
        "date_year + language",
        (
            year
            + " | "
            + language_raw
        ),
        "Date1 from date_year"
    )


    # ========================================================
    # 024
    # ========================================================

    citation_raw = clean(
        row.get(
            "custom_citation",
            ""
        )
    )

    identifier024 = extract_024(
        citation_raw
    )

    if identifier024:

        value = mf(
            "024",
            "8",
            "\\",
            (
                "$a"
                + identifier024
            )
        )

        add_output(
            lines,
            "024",
            record_number,
            excel_row,
            value,
            "EXTRACTED",
            "custom_citation",
            citation_raw,
            "Extracted after PRINT,"
        )

    else:

        value = mf(
            "024",
            "8",
            "\\",
            (
                "$a"
                + PLACEHOLDER_024
            )
        )

        add_output(
            lines,
            "024",
            record_number,
            excel_row,
            value,
            "PLACEHOLDER",
            "custom_citation",
            citation_raw,
            "No PRINT identifier found"
        )


    # ========================================================
    # 035
    # ========================================================

    identifier = clean(
        row.get(
            "identifier",
            ""
        )
    )

    repository = strip_html(
        row.get(
            "repository_original",
            ""
        )
    )

    if identifier:

        if repository:

            body = (
                "$a("
                + repository
                + ")["
                + identifier
                + "]"
            )

        else:

            body = (
                "$a"
                + identifier
            )

        value = mf(
            "035",
            "\\",
            "\\",
            body
        )

        add_output(
            lines,
            "035",
            record_number,
            excel_row,
            value,
            "USED",
            (
                "identifier + "
                "repository_original"
            ),
            (
                identifier
                + " | "
                + repository
            ),
            "Repository identifier"
        )

    else:

        log_skip(
            "035",
            record_number,
            excel_row,
            "identifier",
            "",
            "identifier empty"
        )


    # ========================================================
    # 040
    # ========================================================

    value = mf(
        "040",
        "\\",
        "\\",
        FIXED_040
    )

    add_output(
        lines,
        "040",
        record_number,
        excel_row,
        value,
        "CONSTANT",
        "documentation",
        FIXED_040,
        "Fixed 040"
    )


    # ========================================================
    # 041
    # ========================================================

    value = mf(
        "041",
        "\\",
        "\\",
        (
            "$a"
            + language
        )
    )

    add_output(
        lines,
        "041",
        record_number,
        excel_row,
        value,
        "USED",
        "language",
        language_raw,
        (
            "MARC language code "
            + language
        )
    )


    # ========================================================
    # 043
    # ========================================================

    value = mf(
        "043",
        "\\",
        "\\",
        FIXED_043
    )

    add_output(
        lines,
        "043",
        record_number,
        excel_row,
        value,
        "CONSTANT",
        "documentation",
        FIXED_043,
        "Fixed 043"
    )


    # ========================================================
    # 045
    # ========================================================

    if year:

        value = mf(
            "045",
            "0",
            "\\",
            (
                "$bd"
                + year
            )
        )

        add_output(
            lines,
            "045",
            record_number,
            excel_row,
            value,
            "GENERATED",
            "date_year",
            year,
            "Generated from date_year"
        )


    # ========================================================
    # 046
    # ========================================================

    date_display = clean(
        row.get(
            "date_display",
            ""
        )
    )

    normalized_date = normalize_yyyymmdd(
        date_display
    )

    source046 = "date_display"
    raw046 = date_display

    if not normalized_date:

        raw046 = clean(
            row.get(
                "publication_date",
                ""
            )
        )

        normalized_date = normalize_yyyymmdd(
            raw046
        )

        source046 = "publication_date"

    if normalized_date:

        value = mf(
            "046",
            "\\",
            "\\",
            (
                "$k"
                + normalized_date
            )
        )

        add_output(
            lines,
            "046",
            record_number,
            excel_row,
            value,
            "USED",
            source046,
            raw046,
            "Normalized YYYYMMDD"
        )

    else:

        log_skip(
            "046",
            record_number,
            excel_row,
            (
                "date_display / "
                "publication_date"
            ),
            "",
            "No usable date"
        )


    # ========================================================
    # 049
    # ========================================================

    value = mf(
        "049",
        "\\",
        "\\",
        FIXED_049
    )

    add_output(
        lines,
        "049",
        record_number,
        excel_row,
        value,
        "CONSTANT",
        "documentation",
        FIXED_049,
        "Fixed holdings"
    )


    # ========================================================
    # CREATOR LINKED
    #
    # NEWEST:
    #
    # creator entity 1 -> 100
    # creator entity 2+ -> 700 $eauthor
    #
    # ========================================================

    (
        creator_value,
        creator_source,
        creator_status,
        creator_reason
    ) = get_linked(
        row,
        "creator_linked_new",
        "creator_linked"
    )

    creator_entities = (
        group_authorities(
            creator_value
        )
        if creator_value
        else []
    )


    # ========================================================
    # 100 - FIRST CREATOR ONLY
    # ========================================================

    if creator_entities:

        primary_creator = (
            creator_entities[0]
        )

        (
            body,
            indicator1,
            method
        ) = build_person_body(
            primary_creator,
            row,
            "author"
        )

        value = mf(
            "100",
            indicator1,
            "\\",
            body
        )

        add_output(
            lines,
            "100",
            record_number,
            excel_row,
            value,
            (
                "HEURISTIC"
                if method == "HEURISTIC"
                else creator_status
            ),
            creator_source,
            creator_value,
            (
                creator_reason
                + "; FIRST creator only; "
                  "authority priority "
                  "LC > VIAF > Wikidata"
            )
        )

    else:

        log_skip(
            "100",
            record_number,
            excel_row,
            (
                "creator_linked_new / "
                "creator_linked"
            ),
            "",
            creator_reason
        )


    # ========================================================
    # DISPLAY CREATOR FOR 245 / 246
    # ========================================================

    creator_display = get_creator_display_name(
        row,
        creator_entities
    )


    # ========================================================
    # 245
    # ========================================================

    title = strip_html(
        row.get(
            "title",
            ""
        )
    )

    if title:

        body = (
            "$a"
            + title
        )

        if creator_display:

            body += (
                " /$cby "
                + creator_display
                + "."
            )

        value = mf(
            "245",
            "1",
            "0",
            body
        )

        add_output(
            lines,
            "245",
            record_number,
            excel_row,
            value,
            "USED",
            "title + first creator",
            (
                title
                + " | "
                + creator_display
            ),
            "Title with $c creator"
        )


    # ========================================================
    # 246
    # ========================================================

    alternative = strip_html(
        row.get(
            "title_alternative",
            ""
        )
    )

    if alternative:

        body = (
            "$a"
            + alternative
        )

        if creator_display:

            body += (
                " /$cby "
                + creator_display
                + "."
            )

        value = mf(
            "246",
            "3",
            "\\",
            body
        )

        add_output(
            lines,
            "246",
            record_number,
            excel_row,
            value,
            "USED",
            (
                "title_alternative + "
                "first creator"
            ),
            (
                alternative
                + " | "
                + creator_display
            ),
            "Alternative title"
        )


    # ========================================================
    # 264
    # ========================================================

    sender_place_original = clean(
        row.get(
            "sender_place",
            ""
        )
    )

    (
        sender_place_final,
        sender_place_status,
        sender_place_reason
    ) = get_updated_sender_place(
        sender_place_original
    )

    if (
        sender_place_final
        or year
    ):

        body = ""

        if sender_place_final:

            body += (
                "$a"
                + sender_place_final
            )

            if year:
                body += ","

        if year:

            body += (
                "$c"
                + year
            )

        value = mf(
            "264",
            "\\",
            "0",
            body
        )

        add_output(
            lines,
            "264",
            record_number,
            excel_row,
            value,
            sender_place_status,
            (
                "main.sender_place -> "
                "sender_place_new.xlsx"
            ),
            (
                "ORIGINAL="
                + sender_place_original
                + " | FINAL="
                + sender_place_final
                + " | YEAR="
                + year
            ),
            sender_place_reason
        )

    else:

        log_skip(
            "264",
            record_number,
            excel_row,
            (
                "sender_place + "
                "date_year"
            ),
            sender_place_original,
            "No place or year"
        )


    log_event(
        "sender_place_mapping",
        record_number,
        excel_row,
        sender_place_status,
        "sender_place_new.xlsx",
        sender_place_original,
        sender_place_final,
        sender_place_reason
    )


    # ========================================================
    # sender_place_linked not used in 264
    # ========================================================

    (
        sender_linked_value,
        sender_linked_source,
        sender_linked_status,
        sender_linked_reason
    ) = get_linked(
        row,
        "sender_place_linked_new",
        "sender_place_linked"
    )

    log_event(
        "sender_place_linked",
        record_number,
        excel_row,
        "SKIPPED",
        sender_linked_source,
        sender_linked_value,
        "",
        (
            "264 uses curated "
            "sender_place_new.xlsx mapping; "
            "linked place is not used in 264"
        )
    )


    # ========================================================
    # 300
    # ========================================================

    document_type = clean(
        row.get(
            "document_type",
            ""
        )
    )

    type_field = clean(
        row.get(
            "type_field",
            ""
        )
    )

    is_correspondence = (
        "correspondence"
        in (
            document_type
            + " "
            + type_field
        ).lower()
    )

    if is_correspondence:

        body300 = (
            "$a1 online resource (letter)."
        )

    else:

        body300 = (
            "$a1 online resource."
        )

    value = mf(
        "300",
        "\\",
        "\\",
        body300
    )

    add_output(
        lines,
        "300",
        record_number,
        excel_row,
        value,
        "GENERATED",
        (
            "document_type + "
            "type_field"
        ),
        (
            document_type
            + " | "
            + type_field
        ),
        "Conditional 300"
    )


    # ========================================================
    # 336 337 338
    # ========================================================

    for tag, body in [
        ("336", FIXED_336),
        ("337", FIXED_337),
        ("338", FIXED_338)
    ]:

        value = mf(
            tag,
            "\\",
            "\\",
            body
        )

        add_output(
            lines,
            tag,
            record_number,
            excel_row,
            value,
            "CONSTANT",
            "documentation",
            body,
            "Fixed RDA field"
        )


    # ========================================================
    # 500
    # ========================================================

    location_instance = 0

    for column in [
        "location_1",
        "location_2"
    ]:

        raw = clean(
            row.get(
                column,
                ""
            )
        )

        location = strip_html(
            raw
        )

        if not location:
            continue

        location_instance += 1

        if not location.lower().startswith(
            "location:"
        ):

            location = (
                "Location: "
                + location
            )

        value = mf(
            "500",
            "\\",
            "\\",
            (
                "$a"
                + location
            )
        )

        add_output(
            lines,
            "500",
            record_number,
            excel_row,
            value,
            "USED",
            column,
            raw,
            "Location note",
            location_instance
        )

    if location_instance == 0:

        log_skip(
            "500",
            record_number,
            excel_row,
            (
                "location_1 / "
                "location_2"
            ),
            "",
            "Both empty"
        )


    # ========================================================
    # 505
    # ========================================================

    transcription = strip_html(
        row.get(
            "transcription",
            ""
        )
    )

    if transcription:

        value = mf(
            "505",
            "\\",
            "\\",
            (
                "$a"
                + transcription
            )
        )

        add_output(
            lines,
            "505",
            record_number,
            excel_row,
            value,
            "USED",
            "transcription",
            row.get(
                "transcription",
                ""
            ),
            "Transcription"
        )

    else:

        log_skip(
            "505",
            record_number,
            excel_row,
            "transcription",
            "",
            "Empty"
        )


    # ========================================================
    # 506
    # ========================================================

    embargo = strip_html(
        row.get(
            "embargo_date",
            ""
        )
    )

    if embargo:

        value = mf(
            "506",
            "\\",
            "\\",
            (
                "$a"
                + embargo
            )
        )

        add_output(
            lines,
            "506",
            record_number,
            excel_row,
            value,
            "USED",
            "embargo_date",
            embargo,
            "Restriction note"
        )

    else:

        log_skip(
            "506",
            record_number,
            excel_row,
            "embargo_date",
            "",
            "Empty"
        )


    # ========================================================
    # 520
    # ========================================================

    abstract = strip_html(
        row.get(
            "abstract",
            ""
        )
    )

    if abstract:

        value = mf(
            "520",
            "0",
            "\\",
            (
                "$a"
                + abstract
            )
        )

        add_output(
            lines,
            "520",
            record_number,
            excel_row,
            value,
            "USED",
            "abstract",
            row.get(
                "abstract",
                ""
            ),
            "Abstract"
        )


    # ========================================================
    # 524
    # ========================================================

    citation = strip_html(
        row.get(
            "custom_citation",
            ""
        )
    )

    if citation:

        value = mf(
            "524",
            "\\",
            "\\",
            (
                "$a"
                + citation
            )
        )

        add_output(
            lines,
            "524",
            record_number,
            excel_row,
            value,
            "USED",
            "custom_citation",
            row.get(
                "custom_citation",
                ""
            ),
            "Citation"
        )


    # ========================================================
    # 530
    # ========================================================

    if repository:

        note = (
            "The original letter is available at "
            + repository
        )

        if not note.endswith("."):
            note += "."

        value = mf(
            "530",
            "\\",
            "\\",
            (
                "$a"
                + note
            )
        )

        add_output(
            lines,
            "530",
            record_number,
            excel_row,
            value,
            "USED",
            "repository_original",
            row.get(
                "repository_original",
                ""
            ),
            "Repository note"
        )


    # ========================================================
    # 540
    # ========================================================

    license_value = strip_html(
        row.get(
            "distribution_license",
            ""
        )
    )

    if license_value:

        value = mf(
            "540",
            "\\",
            "\\",
            (
                "$a"
                + license_value
            )
        )

        add_output(
            lines,
            "540",
            record_number,
            excel_row,
            value,
            "USED",
            "distribution_license",
            row.get(
                "distribution_license",
                ""
            ),
            "License"
        )


    # ========================================================
    # 600 PERSON + FAMILY
    # ========================================================

    field600_instance = 0

    person_subject = clean(
        row.get(
            "subject_person_600",
            ""
        )
    )

    if person_subject:

        for entity in group_authorities(
            person_subject
        ):

            field600_instance += 1

            (
                i1,
                i2,
                body,
                source
            ) = build_subject_person(
                entity
            )

            value = mf(
                "600",
                i1,
                i2,
                body
            )

            add_output(
                lines,
                "600",
                record_number,
                excel_row,
                value,
                "USED",
                "subject_person_600",
                person_subject,
                (
                    "Personal subject; "
                    "authority="
                    + source
                ),
                field600_instance
            )


    family = clean(
        row.get(
            "family_linked",
            ""
        )
    )

    if family:

        for entity in group_authorities(
            family
        ):

            field600_instance += 1

            (
                i1,
                i2,
                body,
                source
            ) = build_family_subject(
                entity
            )

            value = mf(
                "600",
                i1,
                i2,
                body
            )

            add_output(
                lines,
                "600",
                record_number,
                excel_row,
                value,
                "USED",
                "family_linked",
                family,
                (
                    "Family subject; "
                    "authority="
                    + source
                ),
                field600_instance
            )

    if field600_instance == 0:

        log_skip(
            "600",
            record_number,
            excel_row,
            (
                "subject_person_600 / "
                "family_linked"
            ),
            "",
            "No 600 data"
        )


    # ========================================================
    # 610
    # ========================================================

    corporate_raw = clean(
        row.get(
            "subject_corporate_610",
            ""
        )
    )

    field610_instance = 0

    if corporate_raw:

        for entity in group_authorities(
            corporate_raw
        ):

            field610_instance += 1

            (
                i1,
                i2,
                body,
                source
            ) = build_corporate_subject(
                entity
            )

            value = mf(
                "610",
                i1,
                i2,
                body
            )

            add_output(
                lines,
                "610",
                record_number,
                excel_row,
                value,
                "USED",
                "subject_corporate_610",
                corporate_raw,
                (
                    "Corporate authority="
                    + source
                ),
                field610_instance
            )


    # ========================================================
    # ORGANIZATION LINKED
    # ========================================================

    organization_raw = clean(
        row.get(
            "organization_linked",
            ""
        )
    )

    organization_entities = (
        group_authorities(
            organization_raw
        )
        if organization_raw
        else []
    )

    receiver_plain = clean(
        row.get(
            "receiver",
            ""
        )
    )

    receiver_org_entity = None

    for org_entity in organization_entities:

        if organization_matches_receiver(
            receiver_plain,
            org_entity
        ):

            receiver_org_entity = (
                org_entity
            )

            break


    if receiver_org_entity is None:

        for entity in organization_entities:

            field610_instance += 1

            lc = get_link(
                entity,
                "LC"
            )

            viaf = get_link(
                entity,
                "VIAF"
            )

            wiki = get_link(
                entity,
                "Wikidata"
            )

            body = (
                "$a"
                + entity[
                    "label"
                ]
            )

            if (
                lc
                and lc["url"]
            ):

                body += (
                    "$0"
                    + lc[
                        "url"
                    ]
                )

            if (
                viaf
                and viaf["url"]
            ):

                body += (
                    "$1"
                    + viaf[
                        "url"
                    ]
                )

            if (
                wiki
                and wiki["url"]
            ):

                body += (
                    "$1"
                    + wiki[
                        "url"
                    ]
                )

                body += "$2wikidata"

            value = mf(
                "610",
                "2",
                "7",
                body
            )

            add_output(
                lines,
                "610",
                record_number,
                excel_row,
                value,
                "USED",
                "organization_linked",
                organization_raw,
                "Organization subject",
                field610_instance
            )

    if field610_instance == 0:

        log_skip(
            "610",
            record_number,
            excel_row,
            (
                "subject_corporate_610 / "
                "organization_linked"
            ),
            "",
            "No 610 data"
        )


    # ========================================================
    # 650
    # ========================================================

    field650_instance = 0

    for item in split_values(
        row.get(
            "subject_topic_lc_650_0",
            ""
        )
    ):

        field650_instance += 1

        value = mf(
            "650",
            "\\",
            "0",
            (
                "$a"
                + item
            )
        )

        add_output(
            lines,
            "650",
            record_number,
            excel_row,
            value,
            "USED",
            "subject_topic_lc_650_0",
            row.get(
                "subject_topic_lc_650_0",
                ""
            ),
            "LCSH",
            field650_instance
        )


    fast_raw = clean(
        row.get(
            "subject_topic_fast_650_7",
            ""
        )
    )

    if fast_raw:

        if "<a " in fast_raw.lower():

            for entity in group_authorities(
                fast_raw
            ):

                field650_instance += 1

                body = (
                    "$a"
                    + entity[
                        "label"
                    ].rstrip(".")
                    + ".$2fast"
                )

                for link in entity[
                    "links"
                ]:

                    if link["url"]:

                        body += (
                            "$0"
                            + link[
                                "url"
                            ]
                        )

                        break

                value = mf(
                    "650",
                    "\\",
                    "7",
                    body
                )

                add_output(
                    lines,
                    "650",
                    record_number,
                    excel_row,
                    value,
                    "USED",
                    "subject_topic_fast_650_7",
                    fast_raw,
                    "FAST",
                    field650_instance
                )

        else:

            for item in split_values(
                fast_raw
            ):

                field650_instance += 1

                value = mf(
                    "650",
                    "\\",
                    "7",
                    (
                        "$a"
                        + item.rstrip(".")
                        + ".$2fast"
                    )
                )

                add_output(
                    lines,
                    "650",
                    record_number,
                    excel_row,
                    value,
                    "USED",
                    "subject_topic_fast_650_7",
                    fast_raw,
                    "FAST",
                    field650_instance
                )

    if field650_instance == 0:

        log_skip(
            "650",
            record_number,
            excel_row,
            (
                "subject_topic_lc_650_0 / "
                "subject_topic_fast_650_7"
            ),
            "",
            "No topical subjects"
        )


    # ========================================================
    # 651
    # ========================================================

    geo_raw = clean(
        row.get(
            "subject_geo_651",
            ""
        )
    )

    if geo_raw:

        for i, entity in enumerate(
            group_authorities(
                geo_raw
            ),
            1
        ):

            geonames = get_link(
                entity,
                "GeoNames"
            )

            wiki = get_link(
                entity,
                "Wikidata"
            )

            chosen = (
                geonames
                or wiki
            )

            label = (
                chosen[
                    "label"
                ]
                if chosen
                else entity[
                    "label"
                ]
            )

            body = (
                "$a"
                + label
            )

            if (
                geonames
                and geonames[
                    "url"
                ]
            ):

                body += (
                    "$1"
                    + geonames[
                        "url"
                    ]
                )

            if (
                wiki
                and wiki[
                    "url"
                ]
            ):

                body += (
                    "$1"
                    + wiki[
                        "url"
                    ]
                )

            if geonames:

                body += "$2geonames"

            elif wiki:

                body += "$2wikidata"

            value = mf(
                "651",
                "\\",
                "7",
                body
            )

            add_output(
                lines,
                "651",
                record_number,
                excel_row,
                value,
                "USED",
                "subject_geo_651",
                geo_raw,
                (
                    "GeoNames uses "
                    "$1 + $2geonames"
                ),
                i
            )

    else:

        log_skip(
            "651",
            record_number,
            excel_row,
            "subject_geo_651",
            "",
            "Empty"
        )


    # ========================================================
    # 653
    # ========================================================

    keywords = split_values(
        row.get(
            "keywords",
            ""
        )
    )

    if keywords:

        for i, item in enumerate(
            keywords,
            1
        ):

            value = mf(
                "653",
                "\\",
                "\\",
                (
                    "$a"
                    + item
                )
            )

            add_output(
                lines,
                "653",
                record_number,
                excel_row,
                value,
                "USED",
                "keywords",
                row.get(
                    "keywords",
                    ""
                ),
                "Keyword",
                i
            )

    else:

        log_skip(
            "653",
            record_number,
            excel_row,
            "keywords",
            "",
            "Empty"
        )


    # ========================================================
    # 655
    # ========================================================

    field655_instance = 0

    for item in split_values(
        row.get(
            "subject_genre_655",
            ""
        )
    ):

        field655_instance += 1

        value = mf(
            "655",
            "\\",
            "7",
            (
                "$a"
                + item
                + "$2lcgft"
            )
        )

        add_output(
            lines,
            "655",
            record_number,
            excel_row,
            value,
            "USED",
            "subject_genre_655",
            row.get(
                "subject_genre_655",
                ""
            ),
            "Genre",
            field655_instance
        )


    for item in split_values(
        row.get(
            "type_field",
            ""
        )
    ):

        if item.lower() == "text":
            continue

        field655_instance += 1

        heading = (
            "Letters (correspondence)"
            if item.lower()
            == "correspondence"
            else item
        )

        value = mf(
            "655",
            "\\",
            "7",
            (
                "$a"
                + heading
                + "$2aat"
            )
        )

        add_output(
            lines,
            "655",
            record_number,
            excel_row,
            value,
            "USED",
            "type_field",
            row.get(
                "type_field",
                ""
            ),
            "AAT type",
            field655_instance
        )

    if field655_instance == 0:

        log_skip(
            "655",
            record_number,
            excel_row,
            (
                "subject_genre_655 / "
                "type_field"
            ),
            "",
            "No genre"
        )


    # ========================================================
    # 700 RECEIVER
    #
    # receiver -> $eCorrespondent
    # ========================================================

    (
        receiver_linked_value,
        receiver_linked_source,
        receiver_linked_status,
        receiver_linked_reason
    ) = get_linked(
        row,
        "receiver_linked_new",
        "receiver_linked"
    )

    receiver_entities = (
        group_authorities(
            receiver_linked_value
        )
        if receiver_linked_value
        else []
    )

    field700_instance = 0


    # ========================================================
    # 710 ORGANIZATION RECEIVER
    # ========================================================

    if receiver_org_entity:

        lc = get_link(
            receiver_org_entity,
            "LC"
        )

        viaf = get_link(
            receiver_org_entity,
            "VIAF"
        )

        wiki = get_link(
            receiver_org_entity,
            "Wikidata"
        )

        chosen = (
            lc
            or viaf
            or wiki
        )

        label = (
            chosen[
                "label"
            ]
            if chosen
            else receiver_org_entity[
                "label"
            ]
        )

        body = (
            "$a"
            + label
            + ",$eCorrespondent"
        )

        if (
            lc
            and lc[
                "url"
            ]
        ):

            body += (
                "$0"
                + lc[
                    "url"
                ]
            )

        if (
            viaf
            and viaf[
                "url"
            ]
        ):

            body += (
                "$1"
                + viaf[
                    "url"
                ]
            )

        if (
            wiki
            and wiki[
                "url"
            ]
        ):

            body += (
                "$1"
                + wiki[
                    "url"
                ]
            )

        value = mf(
            "710",
            "2",
            "\\",
            body
        )

        add_output(
            lines,
            "710",
            record_number,
            excel_row,
            value,
            "USED",
            (
                "receiver + "
                "organization_linked"
            ),
            (
                receiver_plain
                + " | "
                + organization_raw
            ),
            (
                "Receiver identified "
                "as organization"
            )
        )

        log_skip(
            "700",
            record_number,
            excel_row,
            "receiver",
            receiver_plain,
            (
                "Organization receiver "
                "mapped to 710"
            )
        )


    # ========================================================
    # PERSONAL RECEIVER(S) -> 700 $eCorrespondent
    # ========================================================

    else:

        for entity in receiver_entities:

            field700_instance += 1

            (
                body,
                indicator1,
                method
            ) = build_person_body(
                entity,
                row,
                "Correspondent"
            )

            value = mf(
                "700",
                indicator1,
                "\\",
                body
            )

            add_output(
                lines,
                "700",
                record_number,
                excel_row,
                value,
                (
                    "HEURISTIC"
                    if method
                    == "HEURISTIC"
                    else receiver_linked_status
                ),
                receiver_linked_source,
                receiver_linked_value,
                (
                    receiver_linked_reason
                    + "; receiver role="
                      "Correspondent"
                ),
                field700_instance
            )

        if not receiver_entities:

            log_skip(
                "700",
                record_number,
                excel_row,
                (
                    "receiver_linked_new / "
                    "receiver_linked"
                ),
                "",
                receiver_linked_reason
            )


    # ========================================================
    # NEWEST CHANGE:
    #
    # SECONDARY CREATOR(S) -> 700 $eauthor
    #
    # creator_entities[0] already went to 100.
    #
    # Everything from creator_entities[1:] becomes a
    # separate 700.
    # ========================================================

    for secondary_creator in (
        creator_entities[1:]
    ):

        field700_instance += 1

        (
            body,
            indicator1,
            method
        ) = build_person_body(
            secondary_creator,
            row,
            "author"
        )

        value = mf(
            "700",
            indicator1,
            "\\",
            body
        )

        add_output(
            lines,
            "700",
            record_number,
            excel_row,
            value,
            (
                "HEURISTIC"
                if method
                == "HEURISTIC"
                else "SECONDARY_CREATOR"
            ),
            creator_source,
            creator_value,
            (
                "Secondary creator from "
                "creator_linked_new/"
                "creator_linked; first creator "
                "mapped to 100, secondary "
                "creator mapped to 700 "
                "with $eauthor"
            ),
            field700_instance
        )


    # ========================================================
    # 751
    # ========================================================

    (
        receiver_place_value,
        receiver_place_source,
        receiver_place_status,
        receiver_place_reason
    ) = get_linked(
        row,
        "receiver_place_linked_new",
        "receiver_place_linked"
    )

    if receiver_place_value:

        for i, entity in enumerate(
            group_authorities(
                receiver_place_value
            ),
            1
        ):

            geonames = get_link(
                entity,
                "GeoNames"
            )

            wiki = get_link(
                entity,
                "Wikidata"
            )

            chosen = (
                geonames
                or wiki
            )

            label = (
                chosen[
                    "label"
                ]
                if chosen
                else entity[
                    "label"
                ]
            )

            body = (
                "$a"
                + label
            )

            if (
                geonames
                and geonames[
                    "url"
                ]
            ):

                body += (
                    "$1"
                    + geonames[
                        "url"
                    ]
                )

            if (
                wiki
                and wiki[
                    "url"
                ]
            ):

                body += (
                    "$1"
                    + wiki[
                        "url"
                    ]
                )

            if geonames:

                body += "$2geonames"

            elif wiki:

                body += "$2wikidata"

            # Only once
            body += "$4crp"

            value = mf(
                "751",
                "\\",
                "\\",
                body
            )

            add_output(
                lines,
                "751",
                record_number,
                excel_row,
                value,
                receiver_place_status,
                receiver_place_source,
                receiver_place_value,
                (
                    receiver_place_reason
                    + "; $2/$4 once only"
                ),
                i
            )

    else:

        log_skip(
            "751",
            record_number,
            excel_row,
            (
                "receiver_place_linked_new / "
                "receiver_place_linked"
            ),
            "",
            receiver_place_reason
        )


    # ========================================================
    # 773
    # ========================================================

    collection = strip_html(
        row.get(
            "collection",
            ""
        )
    )

    if collection:

        value = mf(
            "773",
            "0",
            "\\",
            (
                "$t"
                + collection
            )
        )

        add_output(
            lines,
            "773",
            record_number,
            excel_row,
            value,
            "USED",
            "collection",
            row.get(
                "collection",
                ""
            ),
            "Collection"
        )


    # ========================================================
    # 856
    # ========================================================

    field856_instance = 0


    transcription_url = clean(
        row.get(
            "transcription_url",
            ""
        )
    )

    if transcription_url:

        field856_instance += 1

        value = mf(
            "856",
            "4",
            "0",
            (
                "$u"
                + transcription_url
            )
        )

        add_output(
            lines,
            "856",
            record_number,
            excel_row,
            value,
            "USED",
            "transcription_url",
            transcription_url,
            "Transcription URL",
            field856_instance
        )


    fulltext_url = clean(
        row.get(
            "fulltext_url",
            ""
        )
    )

    if fulltext_url:

        field856_instance += 1

        value = mf(
            "856",
            "4",
            "0",
            (
                "$u"
                + fulltext_url
                + "$zFull Text"
            )
        )

        add_output(
            lines,
            "856",
            record_number,
            excel_row,
            value,
            "USED",
            "fulltext_url",
            fulltext_url,
            "Full text",
            field856_instance
        )


    calc_url = clean(
        row.get(
            "calc_url",
            ""
        )
    )

    field856_instance += 1

    if calc_url:

        final_calc_url = calc_url
        calc_status = "USED"
        calc_source = "calc_url"

    else:

        final_calc_url = DEFAULT_CALC_URL
        calc_status = "FALLBACK"
        calc_source = "documentation fallback"

    value = mf(
        "856",
        "4",
        "1",
        (
            "$y"
            + CALC_LINK_TEXT
            + "$u"
            + final_calc_url
        )
    )

    add_output(
        lines,
        "856",
        record_number,
        excel_row,
        value,
        calc_status,
        calc_source,
        calc_url,
        (
            "Persistent object URL; "
            "fallback if blank"
        ),
        field856_instance
    )


    # ========================================================
    # hires_url
    # ========================================================

    log_event(
        "hires_url",
        record_number,
        excel_row,
        "SKIPPED",
        "hires_url",
        row.get(
            "hires_url",
            ""
        ),
        "",
        (
            "No MARC target specified "
            "in documentation"
        )
    )


    # ========================================================
    # 956
    # ========================================================

    submitted = clean(
        row.get(
            "date_submitted",
            ""
        )
    )

    if submitted:

        value = mf(
            "956",
            "\\",
            "\\",
            (
                "$a"
                + submitted
            )
        )

        add_output(
            lines,
            "956",
            record_number,
            excel_row,
            value,
            "USED",
            "date_submitted",
            submitted,
            "Workflow date"
        )

    else:

        log_skip(
            "956",
            record_number,
            excel_row,
            "date_submitted",
            "",
            "Empty"
        )


    # ========================================================
    # UNMAPPED / INTERNAL
    # ========================================================

    unmapped_columns = {

        "receiver":
            (
                "Not directly mapped except "
                "for organization receiver detection"
            ),

        "receiver_place":
            (
                "receiver_place_linked_new/"
                "receiver_place_linked used instead"
            ),

        "metadata_status":
            (
                "Internal administrative metadata"
            ),

        "backend_transcription":
            (
                "Internal processing field"
            ),

        "format":
            (
                "No MARC target specified"
            ),

        "disciplines":
            (
                "No MARC target specified"
            ),

        "accessible":
            (
                "No exact MARC target specified"
            ),

        "accessibility_status":
            (
                "No exact MARC target specified"
            ),

        "context_key":
            (
                "Internal repository key"
            ),

        "ctmtime":
            (
                "No MARC target specified"
            )
    }


    for column, reason in (
        unmapped_columns.items()
    ):

        log_event(
            (
                "UNMAPPED_"
                + column
            ),
            record_number,
            excel_row,
            "SKIPPED",
            column,
            row.get(
                column,
                ""
            ),
            "",
            reason
        )


    # ========================================================
    # EMAILS
    # ========================================================

    for i in range(
        1,
        6
    ):

        column = (
            f"author{i}_email"
        )

        log_event(
            (
                "UNMAPPED_"
                + column
            ),
            record_number,
            excel_row,
            "SKIPPED",
            column,
            row.get(
                column,
                ""
            ),
            "",
            (
                "Administrative metadata; "
                "excluded from public MARC"
            )
        )


    return "\n".join(
        lines
    )


# ============================================================
# CONVERT ALL ROWS
# ============================================================

records = []

for record_number, (_, row) in enumerate(
    df.iterrows(),
    start=1
):

    if row.isna().all():
        continue

    records.append(
        convert_row(
            row,
            record_number
        )
    )


# ============================================================
# WRITE MARCEDIT MRK
# ============================================================

with open(
    OUTPUT_MRK,
    "w",
    encoding="utf-8",
    newline="\n"
) as f:

    f.write(
        "\n\n".join(
            records
        )
        + "\n"
    )


# ============================================================
# FIELD LOGS
# ============================================================

for tag, events in LOGS.items():

    safe_tag = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        tag
    )

    pd.DataFrame(
        events
    ).to_csv(
        os.path.join(
            LOG_DIR,
            safe_tag
            + ".csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# MASTER LOG
# ============================================================

master_events = []

for events in LOGS.values():

    master_events.extend(
        events
    )

master_df = pd.DataFrame(
    master_events
)

master_df.to_csv(
    MASTER_LOG,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# SUMMARY LOG
# ============================================================

summary_df = (
    master_df
    .groupby(
        [
            "marc_field",
            "status"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="count"
    )
)

summary_df.to_csv(
    SUMMARY_LOG,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# SENDER PLACE LOG
# ============================================================

sender_place_log = pd.DataFrame(
    LOGS.get(
        "sender_place_mapping",
        []
    )
)

sender_place_log.to_csv(
    "SENDER_PLACE_MAPPING_LOG.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# SPECIAL 100 / 700 AUDIT LOG
# ============================================================

creator_audit_rows = []

for event in (
    LOGS.get(
        "100",
        []
    )
    + LOGS.get(
        "700",
        []
    )
):

    creator_audit_rows.append(
        event
    )

creator_audit_df = pd.DataFrame(
    creator_audit_rows
)

creator_audit_df.to_csv(
    "MARC_100_700_AUDIT.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# ZIP FIELD LOGS
# ============================================================

shutil.make_archive(
    "marc_field_logs",
    "zip",
    LOG_DIR
)


# ============================================================
# REPORT
# ============================================================

print("=" * 80)
print("CONVERSION COMPLETE")
print("=" * 80)

print(
    "Main Excel            :",
    INPUT_FILE
)

print(
    "Sender place mapping  :",
    SENDER_PLACE_FILE
)

print(
    "Sender mappings       :",
    len(
        SENDER_PLACE_LOOKUP
    )
)

print(
    "Input records         :",
    len(df)
)

print(
    "MARC records          :",
    len(records)
)

print(
    "MARC output           :",
    OUTPUT_MRK
)

print(
    "Master log            :",
    MASTER_LOG
)

print(
    "Summary log           :",
    SUMMARY_LOG
)

print(
    "100/700 audit         :",
    "MARC_100_700_AUDIT.csv"
)

print(
    "Sender place log      :",
    "SENDER_PLACE_MAPPING_LOG.csv"
)

print(
    "All individual logs   :",
    "marc_field_logs.zip"
)


# ============================================================
# SHOW SUMMARY
# ============================================================

display(
    summary_df
)


# ============================================================
# PREVIEW FIRST RECORD
# ============================================================

print(
    "\nFIRST GENERATED MARC RECORD:\n"
)

if records:

    print(
        records[0]
    )


# ============================================================
# DOWNLOAD OUTPUTS
# ============================================================

files.download(
    OUTPUT_MRK
)

files.download(
    MASTER_LOG
)

files.download(
    SUMMARY_LOG
)

files.download(
    "MARC_100_700_AUDIT.csv"
)

files.download(
    "SENDER_PLACE_MAPPING_LOG.csv"
)

files.download(
    "marc_field_logs.zip"
)

CONVERSION COMPLETE
Main Excel            : franckenachlass_8.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx
Sender place mapping  : sender_place_new.xlsx
Sender mappings       : 119
Input records         : 324
MARC records          : 324
MARC output           : converted_records_updated.mrk
Master log            : MARC_MASTER_LOG.csv
Summary log           : MARC_LOG_SUMMARY.csv
100/700 audit         : MARC_100_700_AUDIT.csv
Sender place log      : SENDER_PLACE_MAPPING_LOG.csv
All individual logs   : marc_field_logs.zip


,marc_field,status,count
0,001,PLACEHOLDER,324
1,005,GENERATED,324
2,008,GENERATED,324
3,024,EXTRACTED,324
4,035,USED,324
...,...,...,...
64,hires_url,SKIPPED,324
65,sender_place_linked,SKIPPED,324
66,sender_place_mapping,FALLBACK_ORIGINAL,166
67,sender_place_mapping,REPLACED,125



FIRST GENERATED MARC RECORD:

=LDR  00000ckm a2200000 io4500
=001   
=005  20260910175848
=008  260910s1703\\\\flunnn\\\\\\\\\\\\knger\d
=024  8\$aSBB-F_1a_1A_1
=035  \\$a(Staatsbibliothek zu Berlin)[SBB-F_1a_1A_1]
=040  \\$aFTU$beng$erda$cFTU$dFTU
=041  \\$ager
=043  \\$ae-uk-en
=045  0\$bd1703
=046  \\$k17031124
=049  \\$a[elec]FTUU
=100  1\$aFrancke, August Hermann,$d1663-1727,$eauthor$0http://id.loc.gov/authorities/names/n85803927$1https://viaf.org/viaf/66477244$1http://www.wikidata.org/entity/Q58753
=245  10$aAugust Hermann Francke, Edmund Chishull, 24 November 1703 /$cby August Hermann Francke.
=246  3\$aAugust Hermann Francke's letter to Edmund Chishull, 24 November 1703 /$cby August Hermann Francke.
=264  \0$aHalle, Saxony-Anhalt, Germany,$c1703
=300  \\$a1 online resource (letter).
=336  \\$astill image$bsti$2rdacontent
=337  \\$aunmediated$bn$2rdamedia
=338  \\$asheet$bnb$2rdacarrier
=500  \\$aLocation: 51.48298, 11.97202
=500  \\$aLocation: 51.48298, 11.97202
=520  0\$aCorr

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>